# Tatyassar — Cue Classifier Training
**Before running:** Upload `challenges.json` and `solutions.json` using the cell below.

**After running:** Download `model.pt`, `cue_vocab.json`, `thresholds.json` and place them in `models/cue_classifier/`.

In [1]:
# Install dependencies
!pip install -q torch transformers sentence-transformers

In [3]:
# Upload challenges.json and solutions.json
from google.colab import files

print('Upload challenges.json and solutions.json')
uploaded = files.upload()  # select both files at once

import os, shutil
os.makedirs('challenges_solutions_data', exist_ok=True)
for fname in uploaded:
    shutil.move(fname, f'challenges_solutions_data/{fname}')
    print(f'Saved: challenges_solutions_data/{fname}')

Upload challenges.json and solutions.json


Saving challenges.json to challenges.json
Saving solutions.json to solutions.json
Saved: challenges_solutions_data/challenges.json
Saved: challenges_solutions_data/solutions.json


In [4]:
# Full training code
# Text
#  ↓
# MiniLM (embedding)
#  ↓
# 384 vector
#  ↓
# Classifier (NN)
#  ↓
# 11 logits
#  ↓
# Sigmoid → probabilities
#  ↓
# Custom thresholds
#  ↓
# Final predicted labels
import json, re, random, csv
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split
from transformers import AutoTokenizer, AutoModel

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')


# NLP Model (MiniLM embeddings text to vector)
class NLPModel:
    def __init__(self, model_name='sentence-transformers/all-MiniLM-L6-v2'):
        print(f'Loading {model_name} ...')
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(DEVICE)
        self.model.eval()

    def encode(self, text: str) -> torch.Tensor:
        text = text.strip() #Removes spaces:"  hello  " → "hello"
        if not text:
            return torch.zeros(384)
        tokens = self.tokenizer(text, return_tensors='pt', truncation=True).to(DEVICE) #PyTorch tensor (pt) #Cut long text (truncation=True)
        with torch.no_grad():
            out = self.model(**tokens)
            emb = out.last_hidden_state.mean(dim=1)
# "The"   → [0.1, 0.2, 0.1]
# "model" → [0.8, 0.7, 0.9]
# "is"    → [0.2, 0.1, 0.2]
# "very"  → [0.6, 0.5, 0.6]
# "slow"  → [0.9, 0.8, 0.9]
# Feature 1: (0.1 + 0.8 + 0.2 + 0.6 + 0.9) / 5 = 0.52
# Feature 2: (0.2 + 0.7 + 0.1 + 0.5 + 0.8) / 5 = 0.46
# Feature 3: (0.1 + 0.9 + 0.2 + 0.6 + 0.9) / 5 = 0.54
# [0.52, 0.46, 0.54]   ← one vector (in reality: 384 values)
# 5 tokens → 5 vectors NO
# 1 sentence → 1 vector YES
        return emb.cpu().squeeze(0)


# Classifier
class CueClassifier(nn.Module):
    def __init__(self, input_dim=384, num_labels=11, hidden_dim=256, dropout=0.15):# dropout prevents overfitting
# Model cannot rely on specific neurons
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_labels)
        )

    def forward(self, x):
        return self.net(x).clamp(min=-50, max=50)


# Data loading
@dataclass
class CueExample:
    text: str
    label_indices: List[int]


class CueVocab:
    def __init__(self, cues):
        unique = sorted(set(cues))
        self.cues = unique
        self.cue_to_id = {c: i for i, c in enumerate(unique)}

    @property
    def size(self):
        return len(self.cues)


def build_vocab_and_examples(challenges_path, solutions_path):
    challenges = {c['id']: c for c in json.load(open(challenges_path))}
    solutions  = {s['id']: s for s in json.load(open(solutions_path))}

    all_cues = []
    for s in solutions.values():
        for c in s.get('correct_output', {}).get('true_cues', []):
            c = c.strip()
            if len(c) > 2 and any(ch.isalpha() for ch in c):
                all_cues.append(c)

    vocab = CueVocab(all_cues)

    examples = []
    for ex_id, ch in challenges.items():
        sol = solutions.get(ex_id)
        if sol is None:
            continue
        text = ch['input_text']
        true_cues = sol.get('correct_output', {}).get('true_cues', [])
        indices = [vocab.cue_to_id[c] for c in true_cues if c in vocab.cue_to_id]
        examples.append(CueExample(text=text, label_indices=indices))

    return examples, vocab


# Threshold tuning
def tune_thresholds(model, val_loader, num_labels, steps=13):
    model.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(DEVICE)
            probs = torch.sigmoid(model(xb)).cpu()
            all_probs.append(probs)
            all_targets.append(yb)

    all_probs   = torch.cat(all_probs)
    all_targets = torch.cat(all_targets)
    grid = torch.linspace(0.3, 0.9, steps=steps)
    thresholds = []

    for i in range(num_labels):
        p, t = all_probs[:, i], all_targets[:, i]
        if t.sum() == 0:
            thresholds.append(0.5)
            continue
        best_f1, best_th = 0.0, 0.5
        for th in grid:
            preds = (p >= th).float()
            tp = (preds * t).sum().item()
            fp = (preds * (1 - t)).sum().item()
            fn = ((1 - preds) * t).sum().item()
            prec = tp / (tp + fp) if (tp + fp) > 0 else 0
            rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
            if f1 > best_f1:
                best_f1, best_th = f1, float(th)
        thresholds.append(best_th)
        print(f'  Label {i}: best_th={best_th:.2f}, F1={best_f1:.3f}')

    return thresholds


print('Code loaded. Ready to train.')

Using device: cuda
Code loaded. Ready to train.


In [5]:
# Build embeddings
# This takes a few minutes — MiniLM encodes all 3000 examples

root = Path('.')
nlp = NLPModel()

examples, vocab = build_vocab_and_examples(
    root / 'challenges_solutions_data' / 'challenges.json',
    root / 'challenges_solutions_data' / 'solutions.json'
)

print(f'\nExamples : {len(examples)}')
print(f'Cues     : {vocab.size} → {vocab.cues}')

X_list, Y_list = [], []
for ex in examples:
    emb = nlp.encode(ex.text)
    X_list.append(emb.unsqueeze(0))
    y = torch.zeros(vocab.size)
    for idx in ex.label_indices:
        y[idx] = 1.0
    Y_list.append(y.unsqueeze(0))

X = torch.cat(X_list)  # [N, 384]
Y = torch.cat(Y_list)  # [N, L]
print(f'\nX shape: {X.shape}  |  Y shape: {Y.shape}')

Loading sentence-transformers/all-MiniLM-L6-v2 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Examples : 2999
Cues     : 11 → ['aggression', 'anxiety', 'avoidance', 'compulsive behaviors', 'dissociation', 'flashbacks', 'hypervigilance', 'nightmares', 'self-blame', 'substance abuse', 'withdrawal']

X shape: torch.Size([2999, 384])  |  Y shape: torch.Size([2999, 11])


In [9]:
# Train
EPOCHS     = 15
BATCH_SIZE = 32
LR         = 3e-4

dataset  = TensorDataset(X, Y)
val_size = max(1, int(0.1 * len(dataset)))
train_ds, val_ds = random_split(dataset, [len(dataset) - val_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)

model = CueClassifier(input_dim=384, num_labels=vocab.size).to(DEVICE)

label_counts = Y.sum(dim=0)
pos_weight   = (Y.shape[0] - label_counts) / (label_counts + 1e-6)
criterion    = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))
optimizer    = torch.optim.AdamW(model.parameters(), lr=LR)

best_val_loss = float('inf')
best_state    = None

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)

    # Validate
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            val_loss += criterion(model(xb), yb).item() * xb.size(0)

    avg_train = total_loss / (len(dataset) - val_size)
    avg_val   = val_loss / val_size

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if epoch % 5 == 0 or epoch == 1:
        print(f'[Epoch {epoch:3d}/{EPOCHS}]  train={avg_train:.4f}  val={avg_val:.4f}  best_val={best_val_loss:.4f}')

# Load best checkpoint
model.load_state_dict(best_state)
print(f'\nTraining done. Best val loss: {best_val_loss:.4f}')

[Epoch   1/15]  train=0.9125  val=0.9002  best_val=0.9002
[Epoch   5/15]  train=0.8929  val=0.8837  best_val=0.8837
[Epoch  10/15]  train=0.8662  val=0.8661  best_val=0.8661
[Epoch  15/15]  train=0.8468  val=0.8550  best_val=0.8550

Training done. Best val loss: 0.8550


In [10]:
# Tune thresholds
print('[THRESHOLD TUNING]')
thresholds = tune_thresholds(model, val_loader, num_labels=vocab.size)

[THRESHOLD TUNING]
  Label 0: best_th=0.60, F1=0.215
  Label 1: best_th=0.50, F1=0.208
  Label 2: best_th=0.30, F1=0.895
  Label 3: best_th=0.50, F1=0.117
  Label 4: best_th=0.55, F1=0.458
  Label 5: best_th=0.30, F1=0.617
  Label 6: best_th=0.30, F1=0.818
  Label 7: best_th=0.35, F1=0.543
  Label 8: best_th=0.40, F1=0.722
  Label 9: best_th=0.55, F1=0.160
  Label 10: best_th=0.60, F1=0.599


In [11]:
 # Save and download
os.makedirs('models/cue_classifier', exist_ok=True)

torch.save(model.state_dict(), 'models/cue_classifier/model.pt')

json.dump({'cues': vocab.cues},
          open('models/cue_classifier/cue_vocab.json', 'w'), indent=2)

json.dump({'thresholds': thresholds},
          open('models/cue_classifier/thresholds.json', 'w'), indent=2)

print('Saved. Downloading...')

from google.colab import files
files.download('models/cue_classifier/model.pt')
files.download('models/cue_classifier/cue_vocab.json')
files.download('models/cue_classifier/thresholds.json')

Saved. Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>